In [1]:
import os
from http.client import responses

from dataclasses_json import config
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from rich import print

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware, \
    TodoListMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver

# 環境変数を読み込む
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

# 要約生成用のモデルを初期化
model = init_chat_model(
    model="openai/gpt-5.4-mini",
    model_provider="openai",
    # profile={"max_input_tokens": 128_000},
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

In [2]:
from langchain.tools import tool
from pathlib import Path
import subprocess

WORKSPACE = Path("../todo_workspace")


@tool
def list_files(path: str = ".") -> str:
    """
    ワークスペース内の指定ディレクトリ配下のファイルとサブディレクトリを一覧表示する。path は相対パスのみ指定可能。

    Args:
        path: ワークスペース内の相対パス。必ずディレクトリを指す。デフォルトは .（ワークスペースのルートパスを表す）。ワークスペース外にはアクセス不可
     """
    target = (WORKSPACE / path).resolve()
    workspace_root = WORKSPACE.resolve()

    if not str(target).startswith(str(workspace_root)):
        return "エラー：ワークスペース内のディレクトリのみアクセス可能です。"

    if not target.exists():
        return f"エラー：ディレクトリが存在しません: {path}"

    if not target.is_dir():
        return f"エラー：ディレクトリではありません: {path}"

    items = sorted(target.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    if not items:
        return f"ディレクトリが空です: {path}"

    lines = []
    for item in items:
        rel = item.relative_to(workspace_root)
        kind = "[DIR]" if item.is_dir() else "[FILE]"
        lines.append(f"{kind} {rel.as_posix()}")

    return "\n".join(lines)


@tool
def read_file(path: str) -> str:
    """
    ワークスペース内のテキストファイルの内容を読み取る。path は相対パスのみ指定可能。

    Args:
        path: ワークスペース内のファイル名
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "エラー：ワークスペース内のファイルのみ読み取り可能です。"
    if not file_path.exists():
        return f"エラー：ファイルが存在しません: {path}"
    return file_path.read_text(encoding="utf-8")


@tool
def write_file(path: str, content: str) -> str:
    """
    ワークスペース内のテキストファイルに書き込む。path は相対パスのみ指定可能。

    Args:
        path: ワークスペース内のファイル名
        content: ファイルに書き込む内容
    """
    file_path = (WORKSPACE / path).resolve()
    if not str(file_path).startswith(str(WORKSPACE.resolve())):
        return "エラー：ワークスペース内のファイルのみ書き込み可能です。"
    file_path.write_text(content, encoding="utf-8")
    return f"ファイルを書き込みました: {path}"


@tool
def run_tests() -> str:
    """
    ワークスペースで pytest -q を実行し、出力を返す。
    引数は受け取らず、戻り値の形式は
    returncode=0|1
    STDOUT:
    STDERR:
    """
    try:
        result = subprocess.run(
            ["pytest", "-q"],
            cwd=str(WORKSPACE),
            capture_output=True,
            text=True,
            timeout=20,
        )
        return (
            f"returncode={result.returncode}\n\n"
            f"STDOUT:\n{result.stdout}\n\n"
            f"STDERR:\n{result.stderr}"
        )
    except Exception as e:
        return f"テストの実行に失敗しました: {e}"

In [3]:
from rich import print
agent = create_agent(
    model=model,
    tools=[list_files, read_file, write_file, run_tests],
    middleware=[TodoListMiddleware()],
    system_prompt=(
        "あなたはコード修正アシスタントです。複数ステップのタスクに遭遇した場合、まず write_todos を使ってToDoリストを作成してください。"
        "その後、ファイルを読み込み、コードを修正し、テストを実行してください。作業はすべてワークスペース内で行ってください。"
    ),
)
response = agent.invoke({
    "messages":[HumanMessage("ワークスペース内の my_add.py ファイルのコードをテストして修正してください")]
})
print(response)

{
    'messages': [
        HumanMessage(
            content='ワークスペース内の my_add.py ファイルのコードをテストして修正してください',
            additional_kwargs={},
            response_metadata={},
            id='df4345a6-c641-4f04-bbb9-c9520e187a44'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 73,
                    'prompt_tokens': 1491,
                    'total_tokens': 1564,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00144675,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00144675,
                        'upstream_inference_prompt_cost': 0.00111825,
                        'upstream_inference_completions_cost': 0.0003285
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1786070599-cbJtf0K4TnyHwsyQdlNs',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fda1a-83cf-78d0-b92d-5bbdfba18895-0',
            tool_calls=[
                {
                    'name': 'write_todos',
                    'args': {
                        'todos': [
                            {'content': 'my_add.py と関連ファイルを確認する', 'status': 'in_progress'},
                            {'content': '必要に応じて my_add.py を修正する', 'status': 'pending'},
                            {'content': 'テストを実行して結果を確認する', 'status': 'pending'}
                        ]
                    },
                    'id': 'call_ayuLbKxrEKB0MNOwQBEIDZtJ',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 1491,
                'output_tokens': 73,
                'total_tokens': 1564,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content="Updated todo list to [{'content': 'my_add.py と関連ファイルを確認する', 'status': 
'in_progress'}, {'content': '必要に応じて my_add.py を修正する', 'status': 'pending'}, {'content': 
'テストを実行して結果を確認する', 'status': 'pending'}]",
            name='write_todos',
            id='e8b165d8-25c7-41e0-a18e-ee3a4b793cd3',
            tool_call_id='call_ayuLbKxrEKB0MNOwQBEIDZtJ'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 50,
                    'prompt_tokens': 1647,
                    'total_tokens': 1697,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 1024,
                        'cache_write_tokens': 0